### imports & functions

In [1]:
import pandas as pd
import numpy as np
import itertools
import MDAnalysis as mda
import os

In [2]:
def compare_df(contacts_orig,contacts, column='frame_d2t'):

    comparison = contacts_orig.merge(
        contacts[['res_a', 'res_b',column]], 
        on=['res_a', 'res_b'], 
        how='outer', 
        indicator=True,
        suffixes=('_orig', '_new'),
    )

    # Filter for rows that are only in one of the dataframes
    membership_diffs = comparison['_merge'] != 'both'

    is_both = comparison['_merge'] == 'both'
    value_diffs = is_both & comparison.apply(
    lambda row: not np.array_equal(row[column+"_orig"], row[column+'_new']), 
    axis=1
)

    return membership_diffs, value_diffs

### load data for comparison

In [3]:
### run test (optional)

In [4]:
script_content ='''#!/bin/bash
#-----------------------------------------------------------------
# cc_contacts_test_project_MUT16_65_46af39bf2445d01c22c25aadf9d305c4
# 
#-----------------------------------------------------------------

#SBATCH -J "cc_contacts_test_project_MUT16_65_46af39bf2445d01c22c25aadf9d305c4"         	#todo Job name #todo
#SBATCH -o "cc_contacts_test_project_MUT16_65_46af39bf2445d01c22c25aadf9d305c4".%j.out  	#todo Specify stdout output file (%j expands to jobId)
#SBATCH -p bigmem           # Partition/Queue name 'bigmem'
#SBATCH -C broadwell          # select either 'broadwell' or 'skylake'
#SBATCH -N 1           #todo 8 Total number of nodes requested (64 cores/node)
#SBATCH -t 01:00:00     #todo  Run time (hh:mm:ss) - 0.5 hours
#SBATCH -A m2_komet331hpc   # Specify allocation to charge against
#SBATCH --mem=0
#SBATCH --exclusive

source /home/lubaltz/p_3_10/bin/activate

# Commands
export PYTHONPATH=$PYTHONPATH:/home/lubaltz/code/cascade_computing
cd /home/lubaltz/code/cascade_computing
python -c "import bokeh; print('Bokeh version:', bokeh.__version__)"
python -c "import pandas; print('Pandas version:', pandas.__version__)"
python -c "import dask, distributed; print('Dask:', dask.__version__, '| Distributed:', distributed.__version__)"
python3 -u -m src.compute.functions.calc_contacts_opt --path ./examples/data/replica_6/postprocessing --label test_project_MUT16_65_46af39bf2445d01c22c25aadf9d305c4 --output ./examples/data/replica_6/postprocessing/minimal_bondss_bash_ts_0_0_accont_s1_dha_6_eps_2_brks_True_tol_1_buf0.6928203230275509_comp_16_80_sc_True_bb_False_pwi_False_bonds_True_debug_True --cutoff_ha 6 --step 1 --cutoff_mol 0.6928203230275509 --cutoff_eps 2 --breaks True --breaks_tol 1 --n_workers 16 --split_parts 80 --traj_min 0 --traj_max 0 --sidechains True --backbone False --pwi False --q False --bonds True --debug True --c0 10 --ck 20


echo byebye bash '''
with open("./submit_test_job.sh", "w") as f:
    f.write(script_content)

In [5]:
EMAIL="lubaltz@uni-mainz.de"

In [6]:
str_con = '! sbatch --mail-type=ALL --mail-user=' + EMAIL + ' --output=submit_test_job.out' + " submit_test_job.sh"
#os.system(str_con)

In [7]:
#eval
eval_script_content='''#!/bin/bash
#-----------------------------------------------------------------
# eval_cc_test_project_MUT16_65_46af39bf2445d01c22c25aadf9d305c4
# 
#-----------------------------------------------------------------

#SBATCH -J "eval_cc_test_project_MUT16_65_46af39bf2445d01c22c25aadf9d305c4"         	#todo Job name #todo
#SBATCH -o "eval_cc_test_project_MUT16_65_46af39bf2445d01c22c25aadf9d305c4".%j.out  	#todo Specify stdout output file (%j expands to jobId)
#SBATCH -p parallel              	        # Partition/Queue name
#SBATCH -C broadwell #broadwell #skylake          # select either 'broadwell' or 'skylake'
#SBATCH -N 1                     	        #todo 8 Total number of nodes requested (64 cores/node)
#SBATCH -t 00:20:00              	        #todo  Run time (hh:mm:ss) - 0.5 hours
#SBATCH -A m2_komet331hpc             	   # Specify allocation to charge against
##SBATCH --mem=100000
##SBATCH --exclusive

source /home/lubaltz/p_3_10/bin/activate

# Commands
export PYTHONPATH=$PYTHONPATH:/home/lubaltz/code/cascade_computing
cd /home/lubaltz/code/cascade_computing
python3 -m src.compute.functions.eval_contacts --path ./examples/data/replica_6/postprocessing --label test_project_MUT16_65_46af39bf2445d01c22c25aadf9d305c4 --output ./examples/data/replica_6/postprocessing/minimal_bondss_bash_ts_0_0_accont_s1_dha_6_eps_2_brks_True_tol_1_buf0.6928203230275509_comp_16_80_sc_True_bb_False_pwi_False_bonds_True_debug_True --result result_files --pwi False --path_tools /home/lubaltz/code/cascade_computing/src/compute/functions/src/compute/utils --bonds True --n_workers 62


echo byebye bash'''

In [8]:
with open("./submit_eval_test_job.sh", "w") as f:
    f.write(eval_script_content)

In [9]:
str_con_eval = '! sbatch --mail-type=ALL --mail-user=' + EMAIL + ' --output=submit_eval__test_job.out' + " submit_eval_test_job.sh"
#os.system(str_con_eval)

### load data to be tested

In [10]:
#test data - put your test run here
path='../examples/data/replica_6/postprocessing/minimal_bondss_bash_ts_0_0_accont_s1_dha_6_eps_2_brks_True_tol_1_buf0.6928203230275509_comp_16_80_sc_True_bb_False_pwi_False_bonds_True_debug_True'#quatsch_bonds_opt_10_20_ts_0_0_accont_s1_dha_6_eps_2_brks_True_tol_1_buf0.6928203230275509_comp_16_80_sc_True_bb_False_pwi_False_bonds_True_debug_True'
label_tst='test_project_MUT16_65_46af39bf2445d01c22c25aadf9d305c4'

### load data existing contact data to be tested

In [11]:
#ultimate truth contacts - debug case including special bonds (single pair - debug setting)
#path_orig='/home/lubaltz/code/cascade_computing/examples/data/replica_6/postprocessing/unittest_bonds_opt_ts_0_0_accont_s1_dha_6_eps_2_brks_True_tol_1_buf0.6928203230275509_comp_16_80_sc_True_bb_False_pwi_False_bonds_True_debug_True'
#label='test_project_MUT16_65_46af39bf2445d01c22c25aadf9d305c4'

In [12]:
#ultimate truth contacts - debug case including special bonds (10 pairs - debug setting)
path_orig='../examples/data/replica_6/postprocessing/unittest_bonds_opt_splid_10_20_ts_0_0_accont_s1_dha_6_eps_2_brks_True_tol_1_buf0.6928203230275509_comp_20_40_sc_True_bb_False_pwi_False_bonds_True_debug_True'
label='test_project_MUT16_65_46af39bf2445d01c22c25aadf9d305c4'

### load orig intermediate data (if debug)

In [13]:
#atom wise parts
#r=path_orig+"/r_cont_map_0_11.parquet"
#r_df_orig = pd.read_parquet(r)

In [14]:
#atom wise parts
#r_lb=path_orig+"/r_lb_cont_map_0_11.parquet"
#r_lb_orig = pd.read_parquet(r_lb)

In [15]:
#res=path_orig+"/res_cont_map_0_11.parquet"
#res_df_orig = pd.read_parquet(res, columns=["res_a","res_b","type_a","type_b","frame"])

#res_lb=path_orig+"/res_lb_cont_map_0_11.parquet"
#res_df_lb_orig = pd.read_parquet(res_lb,columns=["res_a","res_b","type_a","type_b","frame"])

In [16]:
### load intermediate data

In [17]:
#r_lb=path+"/r_lb_cont_map_0_11.parquet"
#r_lb = pd.read_parquet(r_lb)


In [18]:
####residue wise parts

In [19]:
#res=path+"/res_cont_map_0_11.parquet"
#res_df = pd.read_parquet(res, columns=["res_a","res_b","type_a","type_b","frame"])
#res_df.columns

In [20]:
#res_lb=path+"/res_lb_cont_map_0_11.parquet"
#res_df_lb = pd.read_parquet(res_lb,columns=["res_a","res_b","type_a","type_b","frame"])
#res_df_lb.columns

### compare evaluated contacts

In [21]:
res_dt_orig=path_orig+"/dt_cont_map_0_11.parquet"
res_df_dt_orig = pd.read_parquet(res_dt_orig,columns=["res_a","res_b","type_a","type_b","frame", "frame_dt","frame_d2t", "breaks"])

In [22]:
res_dt=path+"/dt_cont_map_0_11.parquet"
res_df_dt = pd.read_parquet(res_dt,columns=["res_a","res_b","type_a","type_b","frame", "frame_dt","frame_d2t", "breaks"])
res_df_dt.columns

Index(['res_a', 'res_b', 'type_a', 'type_b', 'frame', 'frame_dt', 'frame_d2t',
       'breaks'],
      dtype='object')

In [23]:
res_df_dt

,res_a,res_b,type_a,type_b,frame,frame_dt,frame_d2t,breaks
0,26,1946,GLN,ASN,"[632, 634, 637, 640, 647, 649, 651, 653, 660, ...","[632, 633, 634, 635, 636, 637, 638, 639, 640, ...","[632, 633, 634, 635, 636, 637, 638, 639, 640, ...","[633, 635, 636, 638, 639, 641, 642, 643, 644, ..."
1,36,2020,GLN,TYR,"[28, 176, 177, 178]","[28, 29, 30, 31, 32, 36, 42, 43, 44, 45, 47, 4...","[28, 29, 30, 31, 32, 176, 177, 178]","[29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 4..."
2,40,2020,PRO,TYR,"[2, 147, 153, 404, 418, 422, 423, 424, 425, 48...","[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 1...","[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 1...","[3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, ..."
3,40,2021,PRO,GLU,"[304, 307, 321, 324, 326, 338, 340, 345, 346, ...","[304, 305, 306, 307, 308, 309, 310, 314, 315, ...","[304, 305, 306, 307, 308, 309, 310, 321, 322, ...","[305, 306, 308, 309, 310, 311, 312, 313, 314, ..."
4,41,2020,ALA,TYR,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[71, 79, 81, 89, 96, 120, 122, 123, 288, 760, ..."
...,...,...,...,...,...,...,...,...
284,164,1931,ASN,GLN,"[227, 234, 235, 271, 299, 348, 401, 404, 421, ...","[227, 228, 229, 230, 231, 232, 233, 234, 235, ...","[227, 228, 229, 230, 231, 232, 233, 234, 235, ...","[228, 229, 230, 231, 232, 233, 236, 237, 238, ..."
285,164,1932,ASN,PRO,"[36, 38, 40, 54, 72, 155, 159, 163, 217, 227, ...","[36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 4...","[36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 4...","[37, 39, 41, 42, 43, 44, 45, 46, 47, 48, 49, 5..."
286,164,1933,ASN,ALA,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[77, 78, 79, 80, 81, 82, 85, 86, 87, 88, 89, 9..."
287,164,1935,ASN,ASN,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[72, 85, 86, 88, 89, 92, 96, 102, 103, 105, 10..."


In [24]:
res_df_dt_orig

,res_a,res_b,type_a,type_b,frame,frame_dt,frame_d2t,breaks
0,26,1946,GLN,ASN,"[632, 634, 637, 640, 647, 649, 651, 653, 660, ...","[632, 633, 634, 635, 636, 637, 638, 639, 640, ...","[632, 633, 634, 635, 636, 637, 638, 639, 640, ...","[633, 635, 636, 638, 639, 641, 642, 643, 644, ..."
1,36,2020,GLN,TYR,"[28, 176, 177, 178]","[28, 29, 30, 31, 32, 36, 42, 43, 44, 45, 47, 4...","[28, 29, 30, 31, 32, 176, 177, 178]","[29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 4..."
2,40,2020,PRO,TYR,"[2, 147, 153, 404, 418, 422, 423, 424, 425, 48...","[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 1...","[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 1...","[3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, ..."
3,40,2021,PRO,GLU,"[304, 307, 321, 324, 326, 338, 340, 345, 346, ...","[304, 305, 306, 307, 308, 309, 310, 314, 315, ...","[304, 305, 306, 307, 308, 309, 310, 321, 322, ...","[305, 306, 308, 309, 310, 311, 312, 313, 314, ..."
4,41,2020,ALA,TYR,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[71, 79, 81, 89, 96, 120, 122, 123, 288, 760, ..."
...,...,...,...,...,...,...,...,...
284,164,1931,ASN,GLN,"[227, 234, 235, 271, 299, 348, 401, 404, 421, ...","[227, 228, 229, 230, 231, 232, 233, 234, 235, ...","[227, 228, 229, 230, 231, 232, 233, 234, 235, ...","[228, 229, 230, 231, 232, 233, 236, 237, 238, ..."
285,164,1932,ASN,PRO,"[36, 38, 40, 54, 72, 155, 159, 163, 217, 227, ...","[36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 4...","[36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 4...","[37, 39, 41, 42, 43, 44, 45, 46, 47, 48, 49, 5..."
286,164,1933,ASN,ALA,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[77, 78, 79, 80, 81, 82, 85, 86, 87, 88, 89, 9..."
287,164,1935,ASN,ASN,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[72, 85, 86, 88, 89, 92, 96, 102, 103, 105, 10..."


In [25]:
#compare results - should return an empty series
mem_diffs, val_diffs= compare_df(res_df_dt_orig, res_df_dt)
res_df_dt[val_diffs][['res_a', 'res_b']].values

array([], shape=(0, 2), dtype=int32)

### compare resulting contact table

In [26]:
title='_contacts_only_t.parquet' 
contacts_orig=pd.read_parquet(f'{path_orig}/result_files/{label}{title}', columns=['res_a','res_b','prot_a','prot_b','res_type_a','res_type_b','frame','frame_dt','frame_d2t'])
contacts_orig.columns

Index(['res_a', 'res_b', 'prot_a', 'prot_b', 'res_type_a', 'res_type_b',
       'frame', 'frame_dt', 'frame_d2t'],
      dtype='object')

In [27]:
title='_contacts_only_t.parquet' 
contacts=pd.read_parquet(f'{path}/result_files/{label_tst}{title}', columns=['res_a','res_b','prot_a','prot_b','res_type_a','res_type_b','frame','frame_dt','frame_d2t'])
contacts.columns

Index(['res_a', 'res_b', 'prot_a', 'prot_b', 'res_type_a', 'res_type_b',
       'frame', 'frame_dt', 'frame_d2t'],
      dtype='object')

In [28]:
mem_diffs_t, val_diffs_t= compare_df(contacts_orig, contacts)
contacts[mem_diffs_t]

,res_a,res_b,prot_a,prot_b,res_type_a,res_type_b,frame,frame_dt,frame_d2t


In [29]:
contacts[val_diffs_t]

,res_a,res_b,prot_a,prot_b,res_type_a,res_type_b,frame,frame_dt,frame_d2t


In [30]:
#contacts[contacts['res_b']==2065]
#contacts_orig[contacts_orig['res_b']==2065]

### specific residue pairs

#### frame

In [31]:
#from the original contact data
contacts_frame=np.array([  36,   38,   40,   54,   72,  155,  159,  163,  217,  227,  241,
        250,  267,  272,  273,  276,  287,  297,  301,  316,  322,  331,
        347,  358,  367,  380,  393,  406,  567,  571,  573,  580,  610,
        806,  807,  830,  866,  869,  878,  881,  922,  924,  944,  946,
        972,  982,  992,  999, 1007, 1016, 1017, 1026, 1031, 1032, 1034,
       1049, 1054, 1104, 1150, 1168, 1174, 1175, 1187, 1212, 1245, 1246,
       1269, 1273, 1283, 1284, 1287, 1289, 1298, 1334, 1347, 1352, 1375,
       1384, 1388, 1458, 1462, 1477, 1503, 1549, 1552, 1553, 1563, 1566,
       1568, 1570, 1572, 1573, 1574, 1586, 1592, 1597, 1605, 1628, 1669,
       1704, 1710, 1724, 1747, 1748, 1750, 1751, 1760, 1824, 1826, 1841,
       1879, 1880, 1884, 1886, 1908, 1914, 1930, 1949, 1964, 1982])

In [32]:
contacts[(contacts["res_a"] == 164) & (contacts["res_b"] == 1932)]['frame'].values[0][0:120]

array([  36,   38,   40,   54,   72,  155,  159,  163,  217,  227,  241,
        250,  267,  272,  273,  276,  287,  297,  301,  316,  322,  331,
        347,  358,  367,  380,  393,  406,  567,  571,  573,  580,  610,
        806,  807,  830,  866,  869,  878,  881,  922,  924,  944,  946,
        972,  982,  992,  999, 1007, 1016, 1017, 1026, 1031, 1032, 1034,
       1049, 1054, 1104, 1150, 1168, 1174, 1175, 1187, 1212, 1245, 1246,
       1269, 1273, 1283, 1284, 1287, 1289, 1298, 1334, 1347, 1352, 1375,
       1384, 1388, 1458, 1462, 1477, 1503, 1549, 1552, 1553, 1563, 1566,
       1568, 1570, 1572, 1573, 1574, 1586, 1592, 1597, 1605, 1628, 1669,
       1704, 1710, 1724, 1747, 1748, 1750, 1751, 1760, 1824, 1826, 1841,
       1879, 1880, 1884, 1886, 1908, 1914, 1930, 1949, 1964, 1982])

In [33]:
contacts[(contacts["res_a"] == 164) & (contacts["res_b"] == 1932)]['frame'].values[0][0:200]==contacts_frame

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True])

#### frame dt

In [34]:
contacts_frame_dt=np.array([ 36,  37,  38,  39,  40,  41,  42,  43,  44,  45,  46,  47,  48,
        49,  50,  51,  52,  53,  54,  55,  56,  57,  58,  60,  61,  63,
        64,  65,  66,  68,  72,  73,  74,  76,  77,  78,  79,  80,  82,
        83,  84,  89,  90,  91,  92,  93,  94,  95,  96,  97,  98,  99,
       100, 101, 103, 104, 105, 106, 108, 109, 111, 112, 114, 116, 118,
       120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132,
       133, 134, 135, 136, 138, 139, 141, 142, 150, 151, 152, 154, 155,
       156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
       169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 182,
       183, 184, 185, 186, 187, 188, 189, 191, 192, 193, 197, 198, 199,
       200, 201, 202, 203, 204, 206, 207, 210, 211, 212, 213, 214, 215,
       216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228,
       229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241,
       242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254,
       255, 256, 257, 258, 259, 260, 261, 263, 264, 265, 266, 267, 268,
       269, 270, 271, 272, 273])

In [35]:
contacts[(contacts["res_a"] == 164) & (contacts["res_b"] == 1932)]['frame_dt'].values[0][0:200]

array([ 36,  37,  38,  39,  40,  41,  42,  43,  44,  45,  46,  47,  48,
        49,  50,  51,  52,  53,  54,  55,  56,  57,  58,  60,  61,  63,
        64,  65,  66,  68,  72,  73,  74,  76,  77,  78,  79,  80,  82,
        83,  84,  89,  90,  91,  92,  93,  94,  95,  96,  97,  98,  99,
       100, 101, 103, 104, 105, 106, 108, 109, 111, 112, 114, 116, 118,
       120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132,
       133, 134, 135, 136, 138, 139, 141, 142, 150, 151, 152, 154, 155,
       156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
       169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 182,
       183, 184, 185, 186, 187, 188, 189, 191, 192, 193, 197, 198, 199,
       200, 201, 202, 203, 204, 206, 207, 210, 211, 212, 213, 214, 215,
       216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228,
       229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241,
       242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 25

In [36]:
contacts[(contacts["res_a"] == 164) & (contacts["res_b"] == 1932)]['frame_dt'].values[0][0:200]==contacts_frame_dt

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,

#### frame dt

In [37]:
contacts_frame_d2t=np.array([ 36,  37,  38,  39,  40,  41,  42,  43,  44,  45,  46,  47,  48,
        49,  50,  51,  52,  53,  54,  55,  56,  57,  58,  72,  73,  74,
       155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167,
       168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180,
       217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229,
       230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242,
       243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255,
       256, 257, 258, 259, 260, 261, 267, 268, 269, 270, 271, 272, 273,
       274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286,
       287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299,
       300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312,
       313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 331, 332,
       333, 334, 335, 336, 347, 348, 349, 350, 351, 352, 353, 354, 355,
       356, 357, 358, 359, 367, 368, 369, 370, 371, 372, 373, 374, 375,
       376, 377, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390,
       391, 392, 393, 394, 395])

In [38]:
contacts[(contacts["res_a"] == 164) & (contacts["res_b"] == 1932)]['frame_d2t'].values[0][0:200]

array([ 36,  37,  38,  39,  40,  41,  42,  43,  44,  45,  46,  47,  48,
        49,  50,  51,  52,  53,  54,  55,  56,  57,  58,  72,  73,  74,
       155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167,
       168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180,
       217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229,
       230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242,
       243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255,
       256, 257, 258, 259, 260, 261, 267, 268, 269, 270, 271, 272, 273,
       274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286,
       287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299,
       300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312,
       313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 331, 332,
       333, 334, 335, 336, 347, 348, 349, 350, 351, 352, 353, 354, 355,
       356, 357, 358, 359, 367, 368, 369, 370, 371, 372, 373, 37

In [39]:
contacts[(contacts["res_a"] == 164) & (contacts["res_b"] == 1932)]['frame_d2t'].values[0][0:200]==contacts_frame_d2t

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,

### check for special bonds

In [40]:
title='_contacts_only_t.parquet' 
contacts_orig_sp=pd.read_parquet(f'{path_orig}/result_files/{label}{title}', columns=['res_a','res_b','prot_a','prot_b','res_type_a','res_type_b','frame','frame_dt','frame_d2t', 'cation_pi_t', 'pi_stacking_t', 'hbond_t',
       'salt_bridge_t'])
contacts_orig_sp.columns

Index(['res_a', 'res_b', 'prot_a', 'prot_b', 'res_type_a', 'res_type_b',
       'frame', 'frame_dt', 'frame_d2t', 'cation_pi_t', 'pi_stacking_t',
       'hbond_t', 'salt_bridge_t'],
      dtype='object')

In [41]:
title='_contacts_only_t.parquet' 
contacts_sp=pd.read_parquet(f'{path}/result_files/{label_tst}{title}', columns=['res_a','res_b','prot_a','prot_b','res_type_a','res_type_b','frame','frame_dt','frame_d2t', 'cation_pi_t', 'pi_stacking_t', 'hbond_t',
       'salt_bridge_t'])
contacts_sp.columns

Index(['res_a', 'res_b', 'prot_a', 'prot_b', 'res_type_a', 'res_type_b',
       'frame', 'frame_dt', 'frame_d2t', 'cation_pi_t', 'pi_stacking_t',
       'hbond_t', 'salt_bridge_t'],
      dtype='object')

In [42]:
contacts_sp[['res_a','res_b','cation_pi_t', 'pi_stacking_t', 'hbond_t',
       'salt_bridge_t']]

,res_a,res_b,cation_pi_t,pi_stacking_t,hbond_t,salt_bridge_t
0,1,2065,[],[],[],[]
1,1,2066,[],[],[],[]
2,2,2066,[],[],[],[]
3,2,2068,[],[],[],[]
4,129,2669,[],[],[],[]
...,...,...,...,...,...,...
1129,1931,164,[],[],[],[]
1130,1932,164,[],[],[],[]
1131,1933,164,[],[],[],[]
1132,1935,164,[],[],[],[]


### check for special bonds

#### cation_pi

In [43]:
#check for non-empty cation_pi_t
contacts_sp[contacts_sp['cation_pi_t'].str.len() > 0][['res_a','res_b','cation_pi_t', 'pi_stacking_t', 'hbond_t',
       'salt_bridge_t']]

,res_a,res_b,cation_pi_t,pi_stacking_t,hbond_t,salt_bridge_t
5,130,2669,"[578, 594, 682, 684, 687, 698, 716, 718, 722, ...",[],[],[]
480,103,2055,"[4, 6, 7, 8, 10, 12, 13, 14, 15, 16, 19, 20, 2...",[],[],[]
484,104,2055,"[0, 1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 16, 17, 18...",[],[],[]
485,104,2062,"[273, 274, 287, 293, 297, 299, 300, 305, 309, ...",[],[],[]
534,130,2052,[648],[],[],[]
750,3228,15,"[213, 214, 216, 313, 359, 362, 363, 364, 366, ...",[],[],[]


In [44]:
#check for non-empty cation_pi_t
contacts_orig_sp[contacts_orig_sp['cation_pi_t'].str.len() > 0][['res_a','res_b','cation_pi_t', 'pi_stacking_t', 'hbond_t',
       'salt_bridge_t']]

,res_a,res_b,cation_pi_t,pi_stacking_t,hbond_t,salt_bridge_t
1,130,2669,"[578, 594, 682, 684, 687, 698, 716, 718, 722, ...",[],[],[]
480,103,2055,"[4, 6, 7, 8, 10, 12, 13, 14, 15, 16, 19, 20, 2...",[],[],[]
484,104,2055,"[0, 1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 16, 17, 18...",[],[],[]
485,104,2062,"[273, 274, 287, 293, 297, 299, 300, 305, 309, ...",[],[],[]
534,130,2052,[648],[],[],[]
614,3228,15,"[213, 214, 216, 313, 359, 362, 363, 364, 366, ...",[],[],[]


In [45]:
#check
cation_pi_mem_diffs_t, cation_pi_val_diffs_t= compare_df(contacts_orig_sp, contacts_sp, column="cation_pi_t")
contacts_sp[cation_pi_mem_diffs_t]

,res_a,res_b,prot_a,prot_b,res_type_a,res_type_b,frame,frame_dt,frame_d2t,cation_pi_t,pi_stacking_t,hbond_t,salt_bridge_t


#### pi_stacking

In [46]:
contacts_sp[contacts_sp['pi_stacking_t'].str.len() > 0][['res_a','res_b','cation_pi_t', 'pi_stacking_t', 'hbond_t',
       'salt_bridge_t']]

,res_a,res_b,cation_pi_t,pi_stacking_t,hbond_t,salt_bridge_t
170,8,3233,[],"[99, 106, 523, 861, 953, 963, 984]",[],[]
196,18,3233,[],"[0, 4, 6, 7, 17, 18, 19, 20, 22, 26, 29, 31, 3...",[],[]
264,42,3224,[],"[10, 11, 12, 166, 187]",[],[]
351,63,2002,[],"[95, 614, 622, 819, 981, 1165]",[],[]
370,80,1978,[],[401],[],[]
390,86,1978,[],"[582, 600, 645, 681, 703, 747, 756, 757, 758, ...",[],[]


In [47]:
contacts_orig_sp[contacts_orig_sp['pi_stacking_t'].str.len() > 0][['res_a','res_b','cation_pi_t', 'pi_stacking_t', 'hbond_t',
       'salt_bridge_t']]

,res_a,res_b,cation_pi_t,pi_stacking_t,hbond_t,salt_bridge_t
34,8,3233,[],"[99, 106, 523, 861, 953, 963, 984]",[],[]
60,18,3233,[],"[0, 4, 6, 7, 17, 18, 19, 20, 22, 26, 29, 31, 3...",[],[]
128,42,3224,[],"[10, 11, 12, 166, 187]",[],[]
351,63,2002,[],"[95, 614, 622, 819, 981, 1165]",[],[]
370,80,1978,[],[401],[],[]
390,86,1978,[],"[582, 600, 645, 681, 703, 747, 756, 757, 758, ...",[],[]


In [48]:
#check
pi_stacking_mem_diffs_t, pi_stacking_val_diffs_t= compare_df(contacts_orig_sp, contacts_sp, column="pi_stacking_t")
contacts_sp[pi_stacking_mem_diffs_t]

,res_a,res_b,prot_a,prot_b,res_type_a,res_type_b,frame,frame_dt,frame_d2t,cation_pi_t,pi_stacking_t,hbond_t,salt_bridge_t


#### salt_bridge

In [49]:
contacts_sp[contacts_sp['salt_bridge_t'].str.len() > 0][['res_a','res_b','cation_pi_t', 'pi_stacking_t', 'hbond_t',
       'salt_bridge_t']]

,res_a,res_b,cation_pi_t,pi_stacking_t,hbond_t,salt_bridge_t
482,104,2053,[],[],[],"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
1007,1995,97,[],[],[],"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
1011,1984,98,[],[],[],"[5, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,..."
1019,1995,98,[],[],[],"[249, 251, 301]"


In [50]:
contacts_orig_sp[contacts_orig_sp['salt_bridge_t'].str.len() > 0][['res_a','res_b','cation_pi_t', 'pi_stacking_t', 'hbond_t',
       'salt_bridge_t']]

,res_a,res_b,cation_pi_t,pi_stacking_t,hbond_t,salt_bridge_t
482,104,2053,[],[],[],"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
1007,1995,97,[],[],[],"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
1011,1984,98,[],[],[],"[5, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,..."
1019,1995,98,[],[],[],"[249, 251, 301]"


In [51]:
#check
salt_bridge_mem_diffs_t, salt_bridge_val_diffs_t= compare_df(contacts_orig_sp, contacts_sp, column="salt_bridge_t")
contacts_sp[salt_bridge_mem_diffs_t]

,res_a,res_b,prot_a,prot_b,res_type_a,res_type_b,frame,frame_dt,frame_d2t,cation_pi_t,pi_stacking_t,hbond_t,salt_bridge_t


### Result

In [52]:
#contact data with breaks handeling
bool_all_pairs=len(res_df_dt[mem_diffs])<1
bool_all_frames=len(res_df_dt[val_diffs])<1

if bool_all_pairs and bool_all_frames:
    print('contacts ok')

contacts ok


In [53]:
#evaluation result
bool_all_pairs_t=len(contacts[mem_diffs_t])<1
bool_all_frames_t=len(contacts[val_diffs_t])<1

if bool_all_pairs_t and bool_all_frames_t:
    print('evaluation ok')

evaluation ok


In [54]:
#special bonds

bool_cation_pi=len(contacts_sp[cation_pi_mem_diffs_t])<1
if bool_cation_pi:
    print("cation pi ok")

bool_pi_stacking=len(contacts_sp[pi_stacking_mem_diffs_t])<1
if bool_pi_stacking:
    print("pi stacking ok")

bool_salt_bridge=len(contacts_sp[salt_bridge_mem_diffs_t])<1
if bool_salt_bridge:
    print("salt bridge ok")

cation pi ok
pi stacking ok
salt bridge ok
